# 05 · Badminton — szum, głośność i protokół pomiaru
**Dane rzeczywiste:** cztery serie central/off-center z jednej rakiety, 46 automatycznych kandydatów.
To demonstracja narzędzi, nie pomiar skuteczności wykrywania wad ceramiki.

Wybierz kernel `.venv` / Ping Lab i **Run All**. Ten notebook działa samodzielnie;
nie musisz wcześniej uruchamiać pozostałych. Granice serii i zapowiedzi pochodzą
z `data/raw/badminton/session.json`. `project.json` pozostaje konfiguracją oryginalnych notebooków.
Nowe wyniki trafiają do osobnego katalogu; nagranie źródłowe pozostaje bez zmian.


In [1]:
NOTEBOOK = '05_odpornosc_i_protokol'
from pathlib import Path
import sys, json, hashlib, io, base64
from datetime import datetime
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/pinglab').is_dir()), None)
assert ROOT is not None, 'Otwórz notebook z folderu projektu ceramic-ping-lab.'
sys.path.insert(0, str(ROOT/'src'))
sys.path.insert(0, str(ROOT/'scripts'))
%matplotlib inline
cfg = json.loads((ROOT/'data/raw/badminton/session.json').read_text())
SOURCE = ROOT/'data/raw/badminton'/cfg['audio']
assert SOURCE.is_file(), 'Brak nagrania badminton: nie przełączamy się automatycznie na demo.'
OUT = ROOT/'reports/runs/badminton-notebooks'/NOTEBOOK/datetime.now().strftime('%Y%m%d_%H%M%S_%f')
OUT.mkdir(parents=True)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa, soundfile as sf
from scipy.signal import welch
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from IPython.display import display, Audio, HTML, Markdown
from pinglab.audio import load_audio, Config
from pinglab.recorder import segment_ringdown, ringdown_plot
from pinglab.features import extract_features, feature_columns, decay_curve
from pinglab.learning import make_classifier
from pinglab.series import evaluate_series, paired_folds
from pinglab.embeddings import ASTEncoder
from pinglab.robustness import mix_noise

RUN_AST = False
plt.rcParams.update({'figure.dpi':110, 'axes.spines.top':False, 'axes.spines.right':False})
colors = {'central':'#137f83', 'offcenter':'#c56838'}
r = load_audio(SOURCE, Config(sr=48000, max_seconds=180))
print('Plik:', SOURCE.name, '| SHA256:', hashlib.sha256(SOURCE.read_bytes()).hexdigest())
display(pd.DataFrame([r.qc]).drop(columns=['warnings']))
print('Uwagi QC:', r.qc['warnings'] or 'brak')

Plik: session.wav | SHA256: afc2d308b32b80ead3f80f795707a2059d2e359a33a2d6c20470464d41df7f07


,duration_s,native_sr,channels,selected_channel,peak_dbfs,rms_dbfs,clipping_fraction,dc_offset
0,88.99771,44100,1,0,-17.976912,-45.595115,0.0,8.182004e-09


Uwagi QC: ['Pasmo źródłowe kończy się przy 22050 Hz; resampling go nie odzyska']


## Przygotowanie uderzeń
Ten sam detektor i okna: 25 ms przed + 350 ms po początku, skrócone przed kolejnym zdarzeniem. Etykieta pochodzi z serii. Zdarzenie około 77 s pozostaje niepewne.

In [2]:
rows=[]; clips={}; series_traces={}
for s in cfg['series']:
 z=r.y[int(s['start_s']*r.sr):int(s['end_s']*r.sr)]
 tab,t,db=segment_ringdown(z,r.sr,**cfg['detector'])
 series_traces[s['series_id']]=(z,tab,t,db)
 interactive=ringdown_plot(z,r.sr,tab,t,db)
 interactive.write_html(OUT/(s['series_id']+'_segmentacja.html'),include_plotlyjs=True)
 for k,row in enumerate(tab.itertuples()):
  onset=float(row.onset_s);absolute=s['start_s']+onset
  a=max(0,onset-cfg['pre_s'])
  nxt=float(tab.iloc[k+1].onset_s) if k+1<len(tab) else len(z)/r.sr
  b=min(onset+cfg['feature_tail_s'],nxt-.015,len(z)/r.sr)
  clip=z[int(a*r.sr):int(b*r.sr)]
  eid=f"{s['series_id']}_{k+1:02d}"
  clips[eid]=clip
  uncertain=any(abs(absolute-v)<.05 for v in cfg['uncertain_onsets_s'])
  rows.append(dict(event_id=eid,series_id=s['series_id'],round=s['round'],label=s['label'],
   onset_s=absolute,offset_s=onset-a,window_s=len(clip)/r.sr,
   ringdown_s=row.duration_s,ringdown_status=row.status,uncertain=uncertain,
   peak_dbfs=20*np.log10(max(abs(clip).max(),1e-12)),
   **extract_features(clip,r.sr,onset-a,cfg['fmax'])))
events=pd.DataFrame(rows)
events.to_csv(OUT/'features.csv',index=False)
display(events.groupby(['series_id','label'],sort=False).agg(uderzenia=('event_id','size'),
 szczyt_mediana_dbfs=('peak_dbfs','median'),niepewne=('uncertain','sum')))
display(events[['event_id','onset_s','ringdown_s','ringdown_status','uncertain']])
assert all(not (a<=v<=b) for v in events.onset_s for a,b in cfg['speech_excluded_s'])
assert events.window_s.min()>.05

,,uderzenia,szczyt_mediana_dbfs,niepewne
series_id,label,,,
central_1,central,14,-26.799461,0
offcenter_1,offcenter,12,-34.025085,0
central_2,central,10,-31.644835,0
offcenter_2,offcenter,10,-31.444176,1


,event_id,onset_s,ringdown_s,ringdown_status,uncertain
0,central_1_01,8.525,0.070,zanik zaobserwowany,False
1,central_1_02,9.670,0.100,zanik zaobserwowany,False
2,central_1_03,10.860,0.140,zanik zaobserwowany,False
3,central_1_04,11.885,0.315,zanik zaobserwowany,False
4,central_1_05,12.945,0.170,zanik zaobserwowany,False
5,central_1_06,14.110,1.055,kolejne uderzenie przed zanikiem,False
6,central_1_07,15.165,0.570,zanik zaobserwowany,False
7,central_1_08,16.300,0.215,zanik zaobserwowany,False
8,central_1_09,17.385,0.120,zanik zaobserwowany,False
9,central_1_10,18.610,0.115,zanik zaobserwowany,False


## Posłuchaj kontrolowanej zmiany
Biały szum jest syntetyczny, nie pochodzi z fabryki. Mniejszy SNR oznacza silniejszy szum.

In [3]:
from pinglab.robustness import sensitivity_table
from pinglab.plots import compare_hits
EVENT_ID='central_1_01'
offset=float(events.set_index('event_id').loc[EVENT_ID,'offset_s'])
stats,variants=sensitivity_table(clips[EVENT_ID],r.sr,offset=offset,fmax=cfg['fmax'])
display(stats[['variant','peak_hz','centroid_hz','decay10_s','flatness','clipping_fraction']].round(3))
for key in ['original','gain -12 dB','noise 20 dB','noise 10 dB']:
 print(key);display(Audio(variants[key],rate=r.sr,normalize=False))
compare_hits([variants['original'],variants['noise 10 dB']],r.sr,['Oryginał','Szum 10 dB'],[offset]*2,cfg['fmax']).show(renderer='plotly_mimetype')


,variant,peak_hz,centroid_hz,decay10_s,flatness,clipping_fraction
0,original,890.625,1000.047,0.04,0.014,0.0
1,gain -12 dB,890.625,1000.047,0.04,0.014,0.0
2,noise 20 dB,890.625,2082.958,0.04,0.175,0.0
3,noise 10 dB,890.625,6167.417,0.04,0.605,0.0
4,noise 0 dB,890.625,9396.229,0.06,0.882,0.0


original


gain -12 dB


noise 20 dB


noise 10 dB


## 7. Czy wynik zależy od szumu albo jednego zdarzenia?
Model trenuje na oryginalnych uderzeniach. W odłożonej rundzie zmieniamy głośność albo dodajemy syntetyczny biały szum o zadanym SNR. To test kontrolowany; nie zastępuje nagrań z fabryki. Osobno powtarzamy ocenę bez słabego, niepewnego zdarzenia około 77 s.


In [4]:
robust=[]
for name,snr in [('original',None),('gain -12 dB',None),('white noise 20 dB',20),('white noise 10 dB',10)]:
 altered=events.copy()
 if name!='original':
  for i,row in enumerate(events.itertuples()):
   signal=clips[row.event_id]
   signal=signal*10**(-12/20) if snr is None else mix_noise(signal,np.random.default_rng(100+i).normal(size=len(signal)),snr,seed=i)
   feats=extract_features(signal,r.sr,row.offset_s,cfg['fmax'])
   for key,value in feats.items():altered.loc[i,key]=value
 for family in ['resonance','physics','mfcc']:
  m,_,_=evaluate_series(events,family,test_features=altered)
  m['variant']=name;robust.append(m)
robust=pd.concat(robust,ignore_index=True)
display(robust.pivot_table(index=['family','variant'],columns='test_round',values='balanced_accuracy').round(3))
robust.to_csv(OUT/'robustness.csv',index=False)
uncertain=[]
for family in ['resonance','physics','mfcc','mel']:
 m,_,_=evaluate_series(events.loc[~events.uncertain].reset_index(drop=True),family)
 uncertain.append(m)
uncertain=pd.concat(uncertain,ignore_index=True)
display(uncertain[['family','test_round','n_test','balanced_accuracy','auc']].round(3))
uncertain.to_csv(OUT/'without_uncertain_event.csv',index=False)

test_round                       1     2
family    variant                       
mfcc      gain -12 dB        0.619  0.55
          original           0.619  0.55
          white noise 10 dB  0.589  0.50
          white noise 20 dB  0.488  0.50
physics   gain -12 dB        0.482  0.55
          original           0.482  0.55
          white noise 10 dB  0.500  0.50
          white noise 20 dB  0.464  0.45
resonance gain -12 dB        0.286  0.45
          original           0.286  0.45
          white noise 10 dB  0.500  0.50
          white noise 20 dB  0.464  0.45

,family,test_round,n_test,balanced_accuracy,auc
0,resonance,1,26,0.238,0.238
1,resonance,2,19,0.478,0.522
2,physics,1,26,0.482,0.476
3,physics,2,19,0.589,0.633
4,mfcc,1,26,0.619,0.702
5,mfcc,2,19,0.544,0.678
6,mel,1,26,0.202,0.226
7,mel,2,19,0.550,0.411


## Kolejne nagranie
Stały mikrofon, gain, podparcie i odległość. Nagrywaj obie klasy w każdej rundzie. Zbierz nową sesję po ponownym ustawieniu stanowiska. Najpierw seria z pełnym wybrzmieniem, potem szybsze trafienia. Ten test zmienia sygnał testowy; nie trenujemy tutaj na sztucznie zaszumionych kopiach i nie udajemy dodatkowych niezależnych obiektów.

In [5]:
print('GOTOWE:', NOTEBOOK, '| pliki wynikowe:', OUT)

GOTOWE: 05_odpornosc_i_protokol | pliki wynikowe: /Users/karolsii/dev/ceramic-ping-lab/reports/runs/badminton-notebooks/05_odpornosc_i_protokol/20260928_144910_636078
